In [43]:
!pip install -q ultralytics

In [44]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image as PILImage
from ultralytics import YOLO
model = YOLO('yolov8n.pt')

In [45]:
cap = cv2.VideoCapture("/content/football_jersey_clip.mp4")
fps = cap.get(cv2.CAP_PROP_FPS)
ret,frame = cap.read()
ret2, frame2 = cap.read()

In [46]:
def get_centroids(frame):
  result = model.track(frame, persist=True, verbose=False)
  boxes = result[0].boxes.xyxy.cpu().numpy()
  conf = result[0].boxes.conf.cpu().numpy()
  if result[0].boxes.id is None:
    return []
  ids = result[0].boxes.id.cpu().numpy()
  centroid = []
  for box, c, tid in zip(boxes, conf, ids):
    if c < 0.5:
      continue
    x1, y1, x2, y2 = box
    cx = (x1+x2)/2
    cy = (y1+y2)/2
    centroid.append((cx, cy, tid))
  return centroid

test_centroids = get_centroids(frame)
print("get_centroids output:", test_centroids)

get_centroids output: [(np.float32(403.42545), np.float32(304.4803), np.float32(81.0)), (np.float32(99.70708), np.float32(275.8715), np.float32(82.0)), (np.float32(589.9822), np.float32(353.16367), np.float32(83.0))]


In [47]:
# grab raw boxes once here so jersey-crop/avg-color tests have real data to use
_test_result = model.track(frame, persist=True, verbose=False)
_test_boxes = _test_result[0].boxes.xyxy.cpu().numpy()
print("Sample box for testing:", _test_boxes[0] if len(_test_boxes) else "none detected")

Sample box for testing: [     390.59      284.11      416.26      324.85]


In [48]:
centroid_frame1 = get_centroids(frame)
centroid_frame2 = get_centroids(frame2)

In [49]:
def get_jersey_crop(frame, box):
    x1, y1, x2, y2 = map(int, box)
    height = y2 - y1
    new_y2 = int(y1 + (0.35 * height))
    return frame[y1:new_y2, x1:x2]

test_crop = get_jersey_crop(frame, _test_boxes[0])
print("get_jersey_crop output shape:", test_crop.shape)

get_jersey_crop output shape: (14, 26, 3)


In [50]:
def get_avg_color(crop):
  return crop.mean(axis=(0,1))

test_color = get_avg_color(test_crop)
print("get_avg_color output:", test_color)

get_avg_color output: [      109.7       162.7      140.86]


In [51]:
import math
def get_player_speeds(position_history, fps):
  speeds = []
  for i in range(1, len(position_history)):
    x1, y1 = position_history[i-1]
    x2, y2 = position_history[i]
    distance = math.sqrt((x2-x1)**2 + (y2-y1)**2)
    speeds.append(distance * fps)
  return speeds

test_speeds = get_player_speeds([(0,0),(10,0),(10,10)], fps=30)
print("get_player_speeds output:", test_speeds)

get_player_speeds output: [300.0, 300.0]


In [52]:
def count_sprints(speeds, threshold):
  sprint_count = 0
  was_sprinting = False
  for s in speeds:
    is_sprinting = s > threshold
    if is_sprinting and not was_sprinting:
      sprint_count += 1
    was_sprinting = is_sprinting
  return sprint_count

test_sprints = count_sprints(test_speeds, threshold=100)
print("count_sprints output:", test_sprints)

count_sprints output: 1


In [53]:
def build_player_summary(players_position, player_team, player_distances, fps, sprint_threshold):
  player_summary = {}
  for player_id, pos_history in players_position.items():
    team = player_team.get(player_id, "unknown")
    if not isinstance(team, str):
      team = team.item()
    distance = player_distances[player_id]
    speed = get_player_speeds(pos_history, fps)
    sprint_count = count_sprints(speed, sprint_threshold)
    player_summary[player_id] = {
        "team": team,
        "distance": distance,
        "speed": speed,
        "sprint_count": sprint_count
    }
  return player_summary



In [64]:
model = YOLO('yolov8n.pt')
players_position = {}
tid_color ={}
prev_ids = set()
cap = cv2.VideoCapture("/content/football_jersey_clip.mp4")
frame_count = 0
max_frames = 705

while True:
    ret, frame = cap.read()
    if not ret:
        break
    frame_count += 1
    if frame_count > max_frames:
        break

    centroid = get_centroids(frame)
    result = model.track(frame, persist=True, verbose=False)
    boxes = result[0].boxes.xyxy.cpu().numpy()
    ids = result[0].boxes.id.cpu().numpy() if result[0].boxes.id is not None else []
    for box, tid in zip(boxes, ids):
        crop = get_jersey_crop(frame, box)
        tid_color[tid] = get_avg_color(crop)
    curr_ids = { tid for cx, cy, tid in centroid }

    for cx, cy, tid in centroid:
        if tid in players_position:
          players_position[tid].append((cx, cy))

        else:
            players_position[tid] = [(cx, cy)]
        disappeared = prev_ids - curr_ids
    new_ids = curr_ids - prev_ids
    if new_ids:
      for t1 in new_ids:
          for t2 in disappeared:
              if t1 not in players_position or t2 not in players_position:
                continue
              dist = np.linalg.norm(tid_color[t1] - tid_color[t2])
              if dist < 15:
                players_position[t2] = players_position[t1]+players_position[t2]
                del players_position[t1]
                print("MATCH:", t1, t2)
                break
              print(t1, t2, dist)
    prev_ids = curr_ids

print("Main loop done. Total tracked ids:", len(players_position))

370.0 364.0 21.57750247809585
364.0 368.0 51.290050602707595
MATCH: 367.0 366.0
371.0 367.0 15.646387993314596
367.0 363.0 19.55375632922316
375.0 363.0 63.18515335467183
375.0 374.0 64.66588511445251
372.0 363.0 48.66152404861862
387.0 384.0 15.437850310915117
387.0 374.0 32.55567972321711
372.0 374.0 81.51603426718657
391.0 384.0 43.8574117851599
388.0 391.0 40.80059634543632
391.0 388.0 36.975063638268615
387.0 395.0 22.439931817515273
387.0 395.0 22.25555017266846
395.0 394.0 24.257873113550303
394.0 389.0 36.43709668346751
394.0 391.0 27.982913706476907
394.0 395.0 17.84037142722496
389.0 395.0 50.296932796262205
395.0 374.0 33.955084232070675
389.0 374.0 21.807828655331534
387.0 395.0 27.939102822484294
389.0 395.0 38.29865489711717
MATCH: 392.0 390.0
389.0 390.0 27.655363495362995
390.0 389.0 23.118253365237628
397.0 390.0 27.715338100094232
398.0 390.0 65.2635593864036
397.0 387.0 49.645980440626595
397.0 389.0 21.814891860913576
MATCH: 398.0 387.0
398.0 389.0 49.67776186244891

In [56]:
players_position = {pid: pos for pid, pos in players_position.items() if len(pos) >= 5}

In [57]:
player_distances = {}
for pid, history in players_position.items():
    total = 0
    for i in range(len(history) - 1):
        x1, y1 = history[i]
        x2, y2 = history[i+1]
        d = ((x2-x1)**2 + (y2-y1)**2)**0.5
        total += d
    player_distances[pid] = total
print(player_distances)

{np.float32(147.0): np.float32(239.69958), np.float32(148.0): np.float32(672.4745), np.float32(149.0): np.float32(255.98813), np.float32(150.0): np.float32(374.56448), np.float32(152.0): np.float32(50.15697), np.float32(153.0): np.float32(146.47206), np.float32(154.0): np.float32(350.70035), np.float32(155.0): np.float32(92.47485), np.float32(151.0): np.float32(143.48358), np.float32(156.0): np.float32(498.4309), np.float32(157.0): np.float32(527.1192), np.float32(158.0): np.float32(832.7589), np.float32(159.0): np.float32(356.23962), np.float32(166.0): np.float32(44.976254), np.float32(167.0): np.float32(46.852142), np.float32(168.0): np.float32(466.64975), np.float32(170.0): np.float32(255.68674), np.float32(171.0): np.float32(868.2113), np.float32(172.0): np.float32(171.07257), np.float32(174.0): np.float32(2536.225), np.float32(175.0): np.float32(263.22852), np.float32(178.0): np.float32(533.9087), np.float32(179.0): np.float32(220.31947), np.float32(176.0): np.float32(475.6029), n

In [58]:
colors_only = list(tid_color.values())
ids_only = list(tid_color.keys())
data = np.array(colors_only, dtype=np.float32)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
compactness, labels, centers = cv2.kmeans(data, 2, None, criteria, 10, cv2.KMEANS_RANDOM_CENTERS)
player_team = {tid: labels[i][0] for i, tid in enumerate(ids_only)}
print(player_team)

{np.float32(147.0): np.int32(0), np.float32(148.0): np.int32(1), np.float32(149.0): np.int32(0), np.float32(150.0): np.int32(0), np.float32(151.0): np.int32(0), np.float32(152.0): np.int32(0), np.float32(153.0): np.int32(0), np.float32(154.0): np.int32(1), np.float32(155.0): np.int32(1), np.float32(156.0): np.int32(1), np.float32(157.0): np.int32(1), np.float32(158.0): np.int32(0), np.float32(159.0): np.int32(1), np.float32(166.0): np.int32(1), np.float32(167.0): np.int32(1), np.float32(168.0): np.int32(1), np.float32(170.0): np.int32(0), np.float32(171.0): np.int32(1), np.float32(172.0): np.int32(0), np.float32(173.0): np.int32(1), np.float32(174.0): np.int32(0), np.float32(175.0): np.int32(0), np.float32(176.0): np.int32(0), np.float32(178.0): np.int32(0), np.float32(179.0): np.int32(1), np.float32(181.0): np.int32(0), np.float32(182.0): np.int32(1), np.float32(183.0): np.int32(1), np.float32(185.0): np.int32(0), np.float32(186.0): np.int32(0), np.float32(187.0): np.int32(0), np.floa

In [59]:
team_distance = {}
for pid, dist in player_distances.items():
  if pid not in player_team:
    continue
  team = int(player_team[pid])
  if team not in team_distance:
    team_distance[team] = 0
  team_distance[team] += dist
print(team_distance)

{0: np.float32(8593.955), 1: np.float32(11454.995)}


In [60]:
fps = cap.get(cv2.CAP_PROP_FPS)
build_player_summary(players_position, player_team, player_distances, fps, 30)

{np.float32(147.0): {'team': 0,
  'distance': np.float32(239.69958),
  'speed': [1.152453821388687,
   4.890963902933831,
   1.3515198392833099,
   5.765800323508536,
   7.075194285769812,
   2.6605032254823486,
   3.61629373392275,
   2.329030730611947,
   5.082301339002925,
   4.279533526807774,
   3.013962833756257,
   4.373633179711231,
   5.426850057782069,
   6.460169805378035,
   4.591014641692135,
   11.513418756215072,
   9.798944375631851,
   17.233303658430195,
   25.9991583389905,
   17.85986083497105,
   19.658713222808114,
   44.67724075461602,
   53.943054792821265,
   32.46377283009217,
   41.62767513087332,
   65.27670212348666,
   16.25549615474592,
   19.377962355107243,
   45.359884347923796,
   40.20738529741372,
   47.99719467260293,
   46.45209112983229,
   31.456371992538983,
   28.738299456008964,
   31.322314515503543,
   43.976544619915956,
   50.873288596795994,
   44.663515590370636,
   24.363327716234455,
   22.749740808708204,
   25.255765027811115,
   39

In [61]:
cap_check = cv2.VideoCapture("/content/football_jersey_clip.mp4")
print(cap_check.get(cv2.CAP_PROP_FRAME_COUNT))

705.0
